Using POS-Tagging, Dependency-Parsing on Samsung reviews to find the qualitative aspects of top 5 / 10 features

In [78]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import spacy
from gensim.models import Word2Vec

### Import data

In [79]:
path = r'C:\Users\arnig\Documents\Coding_2024\python_work\UpGrad\DS_C70\Specialization_NLP\Syntactic-Processing\Syntactic-Processing_upgrad\POS Tagging Case Study\Dataset\Samsung.txt'
with open(path, mode='r', encoding='utf-8') as f_open:
    reviews_data = f_open.read()

print(type(reviews_data))
print(len(reviews_data))

<class 'str'>
7488235


### View data

In [80]:
# Splitting into sentences
reviews = reviews_data.split('\n')
reviews[:10]

["I feel so LUCKY to have found this used (phone to us & not used hard at all), phone on line from someone who upgraded and sold this one. My Son liked his old one that finally fell apart after 2.5+ years and didn't want an upgrade!! Thank you Seller, we really appreciate it & your honesty re: said used phone.I recommend this seller very highly & would but from them again!!",
 'nice phone, nice up grade from my pantach revue. Very clean set up and easy set up. never had an android phone but they are fantastic to say the least. perfect size for surfing and social media. great phone samsung',
 'Very pleased',
 'It works good but it goes slow sometimes but its a very good phone I love it',
 'Great phone to replace my lost phone. The only thing is the volume up button does not work, but I can still go into settings to adjust. Other than that, it does the job until I am eligible to upgrade my phone again.Thaanks!',
 'I originally was using the Samsung S2 Galaxy for Sprint and wanted to retu

### Extract features

In [81]:
# Top features (extract the most used noun)
nlp_pos = spacy.load('en_core_web_sm', disable=['parser', 'ner'])
nouns = []
for doc in tqdm(reviews):
    tokens = nlp_pos(doc)
    nouns.extend(token.lemma_ for token in tokens if token.pos_ == 'NOUN')

nouns = pd.Series(nouns).value_counts(normalize= True)

100%|██████████| 46355/46355 [02:15<00:00, 341.40it/s]


In [82]:
# Top 10 features from reviews
top = 10
features = nouns.head(top).index.values
print('Top 10 most discussed features')
features

Top 10 most discussed features


array(['phone', 'battery', 'product', 'time', 'screen', 'card', 'price',
       'problem', 'camera', 'app'], dtype=object)

### Filtering adjective and adverb qualities

In [83]:
# Filtering reviews based on presence of top feature words
feature_reviews = {feature: [doc for doc in reviews if feature in doc] for feature in features}

Filtering functions

In [124]:
def check_adverb(token, feature):
    an = [str(a) for a in list(token.ancestors)]
    if token.pos_ == 'ADV' and feature in an:
        return True
    else:
        return False
    
def check_adjective(token, feature):
    an = [str(a) for a in list(token.ancestors)]
    if token.pos_ == 'ADJ' and feature in an:
        return True
    else:
        return False
    
def extract_verb_dependency(token, feature):    # joins verbs with adjectives and adverbs
    qualities = []
    if token.pos_ in ['VERB', 'AUX'] and feature in [ch.lemma_ for ch in token.children]:   # verb related to feature
        quality = [f'{token.lemma_} {a.lemma_}' for a in token.children                     # qualities related to such verbs
                   if a.dep_ in ['amod', 'acomp', 'advmod', 'advcl', 'neg']]
        if len(quality):
            qualities.extend(quality)
    if len(qualities):
        return qualities

# lists verbs with adjectives and adverbs. But word similarities dont include action semantics
# def extract_verb_dependency(token, feature):    
#     qualities = []
#     if token.pos_ in ['VERB', 'AUX'] and feature in [ch.lemma_ for ch in token.children]:   # verb related to feature
#         qualities = [a.lemma_ for a in token.children                     # qualities related to such verbs
#                    if a.dep_ in ['amod', 'acomp', 'advmod', 'advcl', 'neg', 'det']]
#     if len(qualities):
#         qualities = [token.lemma_] + qualities
#         return qualities


Collect and collate filtered qualities for each document

In [125]:
nlp_dep = spacy.load('en_core_web_sm', disable=['ner'])
feature_qualities = []

for feature, docs in feature_reviews.items():
    
    for doc in tqdm(docs):
        qualities = []
        
        for tok in nlp_dep(doc):

            if tok.text == feature:
                qualities.append(tok.text)

            if check_adverb(tok, feature):                # returns words
                qualities.append(tok.lemma_)

            if check_adjective(tok, feature):                 # returns words
                qualities.append(tok.lemma_)

            verb_dep = extract_verb_dependency(tok, feature)      # return lists
            
            if verb_dep:
                qualities.extend(verb_dep)

        feature_qualities.append(qualities)


100%|██████████| 5061/5061 [00:57<00:00, 87.42it/s] 


In [116]:
len(feature_qualities)

49859

### Model building

Generic word vector on review sentences

In [89]:
reviews_tokens_list = [[tok.text for tok in nlp_pos(doc)] for doc in reviews]

In [109]:
review_model = Word2Vec(
    sentences= reviews_tokens_list,
    epochs= 10,
    window= 5,
    vector_size= 1024,
)

In [110]:
for key in feature_reviews.keys():
    print(f'{key}: {[sim[0] for sim in review_model.wv.most_similar(key)]}')

phone: ['device', 'item', 'cellphone', 'smartphone', 'phones', 'product', 'model', 'unit', 'one', 'Phone']
battery: ['batter', 'Battery', 'Sweet', 'batery', 'span', 'companion', 'batteries', 'housing', 'prolong', 'charge']
product: ['item', 'seller', 'Product', 'cellphone', 'team', 'device', 'article', 'phone', 'vendor', 'customer']
time: ['day', 'friday', 'night', 'schedule', 'vacation', 'Thursday', 'moment', 'date', 'afternoon', 'days']
screen: ['display', 'screens', 'Screen', 'touchscreen', 'button', 'keyboard', 'letters', 'buttons', 'glass', 'pixel']
card: ['micro', 'Card', 'cards', 'chip', 'nano', 'tray', 'Vodafone', 'Minutes', 'kit', 'Leave']
price: ['money', 'cost', 'Price', 'prices', 'priced', 'value', 'prize', 'mid', 'deal', 'looking']
problem: ['issue', 'problems', 'issues', 'trouble', 'difficulty', 'concern', 'hassle', 'downfall', 'complaint', 'complaints']
camera: ['sound', 'Camera', 'display', 'resolution', 'build', 'photo', 'front', 'cam', 'video', 'speakers']
app: ['file

Word vector based on qualities filtered reviews

In [140]:
qualities_model = Word2Vec(
    sentences= feature_qualities,
    epochs= 10,
    window= 5,
    vector_size= 1024,
)

In [141]:
for key in feature_reviews.keys():
    print(f'{key}: {[sim[0] for sim in qualities_model.wv.most_similar(key)]}')

phone: ['old', 'flip', 'unlocked', 'new', 'previous', 'have now', 'work great', 'be new', 'be indeed', 'feel solid']
battery: ['last long', 'last not', 'removable', 'be phone', 'drain fast', 'replaceable', 'spare', 'extra', 'last easily', 'charge when']
product: ['price', 'create when', 'overall', 'happy', 'starter', 'really', 'otherwise', 'perfect', 'own ever', 'especially']
time: ['short', 'long', 'remain most', 'outside', 'hard', 'down', 'again', 'same', 'waste not', 'next']
screen: ['large', 'big', 'bright', 'too', 'be bright', 'clear', 'sharp', 'see not', 'huge', 'small']
card: ['sim', 'insert first', 'concern where', 'go in', 'take first', 'micro', 'correct', 'read where', 'sims', '-']
price: ['product', 'really', 'sufficient', 'worth', 'pretty', 'happy', 'easy', 'create when', 'starter', 'satisfied']
problem: ['at', 'only', 'single', 'previously', 'encounter not', 'have never', 'remain most', 'slight', 'be turn', 'last not']
camera: ['front', 'rear', 'back', 'be well', '13mp', '

Word vectors trained on generic sentences only give the semantic similarity, but word vectors trained on filtered words generate more specific semantic similarity based on the filtering context.